# Background Buzz · Entrenamiento de YOLO

Reentrena YOLO11 nano (preentrenado en COCO) con el dataset de logos de bebidas energéticas (v1: `burn` y `monster`; `redbull` llegará en la v2).

- Spec: `.specify/1_spec.md` D-4 (entrenamiento en Kaggle), §1.8 (dataset) y §1.11 (evaluación).
- Tarea: T-1.3.
- Los pesos y resultados se guardan en `/kaggle/working/`, que forma parte de la salida de cada versión del notebook.
- Requisitos: GPU T4 e Internet activados en **Settings**.

In [1]:
# Comprobar la GPU e instalar la misma versión de Ultralytics que en backend/uv.lock,
# y la librería de Roboflow para descargar el dataset.
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.4.191 roboflow

import roboflow
import ultralytics

ultralytics.checks()
print("roboflow", roboflow.__version__)

Ultralytics 8.4.191 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7190.0/8156.9 GB disk)
roboflow 1.7.3


In [2]:
from pathlib import Path

MODEL = "yolo11n.pt"  # Modelo preentrenado (transfer learning).
EPOCHS = 100  # Vueltas completas al dataset de entrenamiento.
PATIENCE = 20  # Para antes si valid no mejora en 20 épocas.
BATCH = 16  # Imágenes que procesa a la vez la GPU.
IMGSZ = 640  # Mismo tamaño que el dataset de Roboflow.
SAVE_PERIOD = 10  # Guarda un checkpoint cada 10 épocas.

DATASET_VERSION = 1
RUN_NAME = "v1-2-marcas-e100"
EVAL_SPLIT = "test"

OUTPUT_DIR = Path("/kaggle/working")
RUNS_DIR = OUTPUT_DIR / "runs"
DATASET_DIR = OUTPUT_DIR / "datasets" / "v1-2-marcas"

In [3]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

# La clave se lee de Kaggle Secrets: nunca se escribe en el notebook.
api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
rf = Roboflow(api_key=api_key)
project = rf.workspace("backgroundbuzz").project("background-buzz")
dataset = project.version(DATASET_VERSION).download(
    "yolov11", location=str(DATASET_DIR)
)

DATA_YAML = DATASET_DIR / "data.yaml"
print(DATA_YAML.read_text())

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/datasets/v1-2-marcas in yolov11:: 100%|██████████| 1306/1306 [00:00<00:00, 10888.71it/s]

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 2
names: ['burn', 'monster']

roboflow:
  workspace: backgroundbuzz
  project: background-buzz
  version: 1
  license: Private
  url: https://app.roboflow.com/backgroundbuzz/background-buzz/1


In [4]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH,
    imgsz=IMGSZ,
    save_period=SAVE_PERIOD,
    device=0,
    fliplr=0.0,  # Sin volteo: invierte el logo (spec §1.8).
    hsv_h=0.0,  # Sin cambio de tono: el color forma parte del logo.
    seed=0,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.207 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.191 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/datasets/v1-2-marcas/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      2.84G      1.449      1.673      1.172         18        640: 100% ━━━━━━━━━━━━ 36/36 8.4it/s 4.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 5.4it/s 0.4s
                   all         49         49       0.93      0.958      0.968      0.649

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      2.84G      1.325     0.9863      1.113         16        640: 100% ━━━━━━━━━━━━ 36/36 8.8it/s 4.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 6.0it/s 0.3s
                   all         49         49      0.961      0.854      0.955      0.636

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      2.84G      1.236      0.827      1.079         18        640: 100% ━━━━━━━━━━━━ 36/36 8.6it/s 4.2s
                 Class     Images  Instances      Box(P          R

In [5]:
best = YOLO(str(RUNS_DIR / RUN_NAME / "weights" / "best.pt"))
metrics = best.val(
    data=str(DATA_YAML),
    split=EVAL_SPLIT,
    imgsz=IMGSZ,
    device=0,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}-eval",
    exist_ok=True,
)

print(f"Conjunto: {EVAL_SPLIT}")
print(f"mAP50: {metrics.box.map50:.3f} · mAP50-95: {metrics.box.map:.3f}")
for pos, cls in enumerate(metrics.box.ap_class_index):
    p, r, map50, map5095 = metrics.box.class_result(pos)
    print(
        f"{best.names[cls]:>14}  P={p:.3f}  R={r:.3f}  mAP50={map50:.3f}  mAP50-95={map5095:.3f}"
    )

Ultralytics 8.4.191 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,294 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1263.1±269.1 MB/s, size: 37.1 KB)
val: Scanning /kaggle/working/datasets/v1-2-marcas/test/labels... 29 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 29/29 1.2Kit/s 0.0s
val: New cache created: /kaggle/working/datasets/v1-2-marcas/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.5it/s 0.8s
                   all         29         58      0.998          1      0.995       0.77
                  burn         29         29      0.996          1      0.995      0.776
               monster         29         29          1          1      0.995      0.765
Speed: 4.5ms preprocess, 7.3ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /kaggle/working/runs/v1-2-marcas-e100-